# 10 — Manufacturing evidence and external-review audit

This is the review entry point for parts, printing and provenance. It reads the
current export, shows what can be independently checked offline, and separates
estimated mass, geometric fit, arranged beds, slicing and physical validation.
Missing evidence remains visible. No printer connection, upload or print start
is performed.


In [ ]:
import os
for key in ("OPENBLAS_NUM_THREADS", "OMP_NUM_THREADS", "MKL_NUM_THREADS"):
    os.environ[key] = "1"
os.environ.setdefault("MPLCONFIGDIR", "/tmp/uavdesign-notebook-matplotlib")
"""Locate the repo root so `import uavdesign` works without `pip install -e .`."""
import sys
from pathlib import Path

REPO_ROOT = Path.cwd()
while not (REPO_ROOT / "uavdesign").is_dir() and REPO_ROOT != REPO_ROOT.parent:
    REPO_ROOT = REPO_ROOT.parent
sys.path.insert(0, str(REPO_ROOT))

PLATFORM_DIR = REPO_ROOT / "platforms" / "survey_wing"
FIG_DIR = REPO_ROOT / "outputs" / "figures" / "survey_wing"
FIG_DIR.mkdir(parents=True, exist_ok=True)

import yaml
import numpy as np
import matplotlib.pyplot as plt

import json
import pandas as pd
from IPython.display import display, Markdown, Image
from uavdesign import design as D
from uavdesign import survey as sv
from uavdesign.atmosphere import G
studies = yaml.safe_load((PLATFORM_DIR / "notebook_studies.yaml").read_text())
d = D.run(solve_stability=False)
s = d.design["survey"]
cam = sv.Camera.from_dict(s["camera"])
from dataclasses import replace
acquisition_cam = replace(cam, max_fps=min(cam.max_fps, s["acquisition_rate_cap_hz"]))
pl = d.design["planform"]
req = d.design["requirements"]

def save(fig, name):
    fig.tight_layout()
    path = FIG_DIR / name
    fig.savefig(path, dpi=140)
    display(Image(filename=str(path)))
    plt.close(fig)


## Reproduction and evidence identity

Source inputs live in versioned YAML; the tables below fingerprint the exact
inputs loaded for this run. The implementation hash map is also retained by the
`Design` result and exported with the dossier. The generated calibration record's
`measured` label means CAD-derived mass, **not scale measurements**.

For a fresh environment, install the repository's locked dependencies, run
`python scripts/build_parts.py`, then `python -m uavdesign.printing plan`.
Execute notebooks from their platform directory or the repository root. The
review exporter preserves executed notebooks and embedded figures so a reader
can inspect the findings without installing CAD software. Re-running the code
still requires the declared Python dependencies and source tree.

A SHA-256 match proves that a file is the same file recorded by a manifest. It
cannot prove that a model represents the physical aircraft. The CAD manifest's
result signature is displayed as identity evidence; dimensions and masses below
are cross-checks rather than a claim of complete source-to-CAD provenance.


In [ ]:
import hashlib
fingerprints = [{"input_file": name, "sha256": sha} for name, sha in sorted(d.input_hashes.items())]
display(pd.DataFrame(fingerprints))
print("Implementation Python files fingerprinted:", len(d.source_hashes))
manifest_path = REPO_ROOT / "outputs/parts/manifest.json"
manifest = json.loads(manifest_path.read_text()) if manifest_path.exists() else None
if manifest is None:
    print("No parts manifest. Run: python scripts/build_parts.py")
else:
    print("Manifest SHA-256:", hashlib.sha256(manifest_path.read_bytes()).hexdigest())
    print("CAD result signature:", manifest["result_signature"])
    print("CAD maturity:", manifest["maturity"])
    hashes = []
    for relative, expected in sorted(manifest["file_sha256"].items()):
        path = (manifest_path.parent / relative).resolve()
        inside = path.is_relative_to(manifest_path.parent.resolve())
        exists = inside and path.is_file()
        hashes.append({"file":relative, "present":exists,
                       "sha256_matches":exists and hashlib.sha256(path.read_bytes()).hexdigest() == expected})
    hash_frame = pd.DataFrame(hashes)
    display(hash_frame.groupby(["present", "sha256_matches"]).size().rename("file_count").to_frame())
    if not hash_frame["sha256_matches"].all():
        display(hash_frame[~hash_frame["sha256_matches"]])
        raise ValueError("CAD files failed manifest identity checks; review cannot use them as a coherent export")


## Scale, manufacturing estimates and design consistency

The printer envelope is a build-volume constraint; the assembled span is a
separate aircraft dimension. Pieces are split along span and sometimes chord to
fit while preserving layer direction. The part-height chart indicates tall prints,
not stability on the bed, cooling adequacy or print duration.

Mass by profile comes from CAD deposition assumptions. Supports, brim, waste,
adhesive, process-dependent foaming and hardware tolerances require separate
accounting. Nominal wall thickness is perimeters times extrusion width; actual
thin-wall toolpaths must be inspected in the slicer. The source STL envelope,
assembly geometry and finished-part dimension are three different kinds of data.


In [ ]:
if manifest is not None:
    parts = pd.DataFrame(manifest["parts"])
    display(parts[["part", "group", "profile", "material", "mass_g", "footprint_mm", "height_mm", "fits"]])
    groups = parts.groupby("profile").agg(parts=("part", "size"), mass_estimate_g=("mass_g", "sum"), tallest_mm=("height_mm", "max"))
    display(groups)
    fig, axes = plt.subplots(1, 2, figsize=(12, 5))
    groups["mass_estimate_g"].sort_values().plot.barh(ax=axes[0], color="C0")
    axes[0].set(xlabel="Estimated deposited part mass [g]", ylabel="Print profile")
    for index, (profile, frame) in enumerate(parts.groupby("profile")):
        axes[1].scatter([index]*len(frame), frame.height_mm, s=35, label=profile)
    axes[1].axhline(manifest["printer"]["max_z_mm"], ls="--", color="C3", label="Printer height limit")
    axes[1].set(xticks=range(len(groups)), xticklabels=list(groups.index), ylabel="Print-oriented height [mm]")
    axes[1].tick_params(axis="x", rotation=50)
    axes[1].grid(axis="y", alpha=.25)
    save(fig, "10_manufacturing_scale.png")
    reference = D.airframe_geometry(d)
    # solve_stability=False supplies placeholder trim, so exclude trim/NP/CG here.
    geometry_keys = ["root_chord_mm", "tip_chord_mm", "half_span_mm", "le_sweep_deg", "pod_width_mm", "pod_height_mm", "pod_length_mm"]
    geometry_checks = [{"quantity":key, "current_analytic_input":reference[key], "CAD_recorded":manifest["design"][key],
                        "difference":manifest["design"][key]-reference[key]} for key in geometry_keys]
    display(pd.DataFrame(geometry_checks))
    display(pd.DataFrame([{"sizing_mass_g":d.selected.auw_kg*1000, "CAD_balance_mass_g":manifest["balance"]["auw_g"],
                           "difference_percent":100*(manifest["balance"]["auw_g"]/(d.selected.auw_kg*1000)-1),
                           "printed_mass_estimate_g":parts.mass_g.sum()}]))
    print("CAD geometry is rounded in the manifest. Agreement is expected within export precision, not exact decimal equality.")


## Current bed layout and its remaining limitations

The plan reader verifies every source STL, the exact part set, project settings,
and checksums of the generated review files. The diagram uses the actual stored
**convex footprint** of each placed mesh; hollow rings and concave pockets remain
filled in this conservative model. Profile-specific settings isolate plates. The
area percentage measures projected hull occupation only and is not a print-time,
filament-use or global packing-optimality claim.

Dashed process halos reserve space; actual supports and brims can still differ.
Layer-by-layer printing is required on multiple-object beds. Nozzle installation,
material calibration, bed adhesion and a full sliced-toolpath inspection remain
separate decisions. A final bed review does not authorize a physical print start.


In [ ]:
from uavdesign.printing.project import PrintProject
from matplotlib.patches import Polygon
project = PrintProject(REPO_ROOT)
plan = None
if manifest is not None:
    try:
        plan = project.read_plan()
    except (ValueError, FileNotFoundError) as error:
        print("Current print-plan evidence unavailable:", error)
        print("Regenerate offline with: python -m uavdesign.printing plan")
if plan is not None:
    print("Plan identity:", project.plan_id(), "State:", plan["artifact_state"], "Sliced:", plan["sliced"])
    print("Packing method:", plan["optimization"])
    bed_x, bed_y, bed_z = plan["project"]["printer"]["bed_mm"]
    layout = plan["project"]["layout"]
    print("Required process halo per side [mm]:", layout["process_clearance_mm"])
    print("Additional inter-part gap [mm]:", layout["part_spacing_mm"])
    plate_rows = []
    columns = 3
    rows = int(np.ceil(len(plan["plates"])/columns))
    fig, axes = plt.subplots(rows, columns, figsize=(12, rows*3.5), squeeze=False)
    for ax, plate in zip(axes.flat, plan["plates"]):
        occupied = 0.0
        for index, placement in enumerate(plate["placements"]):
            xy = np.asarray(placement["placed_polygon_mm"])
            halo = np.asarray(placement["reserved_polygon_mm"])
            occupied += .5*abs(np.dot(xy[:,0], np.roll(xy[:,1], 1))-np.dot(xy[:,1], np.roll(xy[:,0], 1)))
            ax.add_patch(Polygon(halo, fill=False, linestyle="--", edgecolor=".65", linewidth=.65))
            ax.add_patch(Polygon(xy, facecolor=f"C{index%10}", alpha=.6, edgecolor=".2", linewidth=.6))
            centre = xy.mean(axis=0)
            ax.text(*centre, str(index+1), ha="center", va="center", fontsize=8)
        ax.set(xlim=(0,bed_x), ylim=(0,bed_y), aspect="equal", title=f"{plate['id']} · {plate['profile']}")
        ax.set_xlabel("x [mm]"); ax.set_ylabel("y [mm]")
        plate_rows.append({"bed":plate["id"], "profile":plate["profile"], "part_count":len(plate["placements"]),
                           "projected_hull_occupation_percent":100*occupied/(bed_x*bed_y),
                           "parts_in_number_order":", ".join(p["part"] for p in plate["placements"])})
    for ax in list(axes.flat)[len(plan["plates"]):]: ax.set_visible(False)
    save(fig, "10_print_bed_atlas.png")
    display(pd.DataFrame(plate_rows))
    print("Unvalidated manufacturing items:")
    for item in plan["outstanding_validation"]: print("-", item)


## External-agent review protocol

1. Record the dossier/source revision, notebook execution status and manifest
   hashes. Reproduce equations in notebooks 01, 03, 04 and 09 from the YAML;
   distinguish shared-assumption agreement from independent evidence.
2. Inspect assembly views and STEP, then connect each issue to a named part,
   source key, model function and observable result. Print coordinates are not
   aircraft coordinates; verify datums before comparing dimensions.
3. Review load path, trim at realizable CG, launch thrust, wind/imaging limits,
   rail transients, thermal path and capture timing. Numerical convergence,
   successful tessellation and a valid JSON handoff do not close these issues.
4. Treat missing measured polar, material coupons, joined-structure tests,
   motor/propeller curves, installed harness tests, scale/balance measurements,
   calibration and flight/survey trials as missing evidence. Do not manufacture
   test results to complete a review.
5. Return findings with: **severity; assertion; file/key or part; calculation or
   reproduction command; evidence; proposed change; and acceptance test**.
   Separate a documentation correction, model defect and physical design change.
   Prefer a small reproducible counterexample over an untraceable recommendation.

This notebook intentionally performs no procurement, flight-plan deployment,
printer upload or print start. The user's approval is required before starting
any print; the exported layouts are the artifact to review first.
